Core Extraction Functions

In [14]:
from pathlib import Path
import zipfile
import tarfile
import gzip
import bz2
import lzma
import shutil
import os


def is_archive(file_path):
    """Check if a file is a supported archive format."""
    archive_extensions = {
        '.zip', '.tar', '.gz', '.bz2', '.xz', 
        '.tgz', '.tbz2', '.txz', '.tar.gz', 
        '.tar.bz2', '.tar.xz', '.rar', '.7z'
    }
    
    file_lower = str(file_path).lower()
    return any(file_lower.endswith(ext) for ext in archive_extensions)


def extract_archive(archive_path, extract_to):
    """Extract an archive file to a specified directory."""
    archive_path = Path(archive_path)
    extract_to = Path(extract_to)
    
    try:
        extract_to.mkdir(parents=True, exist_ok=True)
        
        suffix_lower = archive_path.suffix.lower()
        
        # ZIP files
        if suffix_lower == '.zip':
            with zipfile.ZipFile(archive_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
            return True
        
        # TAR files (including .tar.gz, .tar.bz2, .tar.xz)
        elif '.tar' in archive_path.suffixes or suffix_lower in {'.tgz', '.tbz2', '.txz'}:
            with tarfile.open(archive_path, 'r:*') as tar_ref:
                tar_ref.extractall(extract_to)
            return True
        
        # GZIP files (single file compression)
        elif suffix_lower == '.gz' and '.tar' not in str(archive_path):
            output_file = extract_to / archive_path.stem
            with gzip.open(archive_path, 'rb') as f_in, open(output_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
            return True
        
        # BZIP2 files (single file compression)
        elif suffix_lower == '.bz2' and '.tar' not in str(archive_path):
            output_file = extract_to / archive_path.stem
            with bz2.open(archive_path, 'rb') as f_in, open(output_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
            return True
        
        # XZ files (single file compression)
        elif suffix_lower == '.xz' and '.tar' not in str(archive_path):
            output_file = extract_to / archive_path.stem
            with lzma.open(archive_path, 'rb') as f_in, open(output_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
            return True
        
        else:
            print(f"  ⚠️  Unsupported archive format: {archive_path.name}")
            return False
            
    except Exception as e:
        print(f"  ❌ Error extracting {archive_path.name}: {str(e)}")
        return False


def get_archive_stem(archive_path):
    """Get the name without archive extensions (handles .tar.gz, .tar.bz2, etc.)"""
    archive_path = Path(archive_path)
    name = archive_path.name
    
    # Handle compound extensions like .tar.gz
    compound_extensions = ['.tar.gz', '.tar.bz2', '.tar.xz']
    for ext in compound_extensions:
        if name.lower().endswith(ext):
            return name[:-len(ext)]
    
    return archive_path.stem


def extract_nested_archives(directory, protected_files=None, current_depth=0, max_depth=10):
    """
    Recursively find and extract all archives in a directory, then delete the archive files.
    
    Args:
        directory: Directory to search for archives
        protected_files: Set of file paths that should NOT be deleted
        current_depth: Current recursion depth
        max_depth: Maximum recursion depth to prevent infinite loops
    """
    directory = Path(directory)
    
    if protected_files is None:
        protected_files = set()
    
    if current_depth > max_depth:
        print(f"  ⚠️  Maximum recursion depth ({max_depth}) reached at {directory}")
        return
    
    if not directory.exists():
        return
    
    # Collect all archives first (to avoid modifying directory while iterating)
    archives_to_process = []
    
    try:
        for item in directory.rglob('*'):
            if item.is_file() and is_archive(item):
                archives_to_process.append(item)
    except PermissionError:
        print(f"  ⚠️  Permission denied: {directory}")
        return
    
    # Process each archive
    for archive_path in archives_to_process:
        if not archive_path.exists():  # May have been deleted already
            continue
        
        # Check if this file is protected
        is_protected = archive_path.resolve() in protected_files
        
        # Extract to a folder in the same location as the archive
        folder_name = get_archive_stem(archive_path)
        extract_folder = archive_path.parent / folder_name
        
        indent = "  " * current_depth
        relative_path = archive_path.relative_to(directory) if archive_path.is_relative_to(directory) else archive_path.name
        print(f"{indent}📦 Extracting: {relative_path} → {folder_name}/")
        
        # Extract the archive
        if extract_archive(archive_path, extract_folder):
            # Delete the archive file after successful extraction (only if not protected)
            if not is_protected:
                try:
                    archive_path.unlink()
                    print(f"{indent}   🗑️  Deleted: {archive_path.name}")
                except Exception as e:
                    print(f"{indent}   ⚠️  Could not delete {archive_path.name}: {e}")
            else:
                print(f"{indent}   🔒 Protected: {archive_path.name} (kept)")
            
            # Recursively process the extracted folder for nested archives
            extract_nested_archives(extract_folder, protected_files, current_depth + 1, max_depth)


def extract_single_archive(archive_path, output_dir=None):
    """
    Extract a single archive file and all nested archives within it.
    The main archive file is NOT deleted, only nested archives are removed.
    
    Args:
        archive_path: Path to the archive file
        output_dir: Output directory (if None, creates folder next to archive)
    """
    archive_path = Path(archive_path)
    
    if not archive_path.is_absolute():
        archive_path = archive_path.expanduser()
    else:
        archive_path = archive_path.resolve()
    
    if not archive_path.exists():
        raise FileNotFoundError(f"Archive does not exist: {archive_path}")
    
    if not archive_path.is_file():
        raise ValueError(f"Path is not a file: {archive_path}")
    
    if not is_archive(archive_path):
        raise ValueError(f"File is not a supported archive: {archive_path}")
    
    # Determine output directory
    if output_dir is None:
        output_dir = archive_path.parent / get_archive_stem(archive_path)
    else:
        output_dir = Path(output_dir).expanduser().resolve()
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 70)
    print("🚀 SINGLE ARCHIVE EXTRACTOR")
    print("=" * 70)
    print(f"📂 Input:  {archive_path}")
    print(f"📂 Output: {output_dir}")
    print(f"🔒 Main archive will be preserved")
    print("=" * 70)
    
    # Protect the main archive from deletion
    protected_files = {archive_path.resolve()}
    
    # Extract the main archive
    print(f"📦 Extracting main archive: {archive_path.name}")
    if extract_archive(archive_path, output_dir):
        print(f"✅ Main archive extracted successfully")
        
        # Extract all nested archives (but don't delete the main one)
        print(f"\n🔍 Searching for nested archives...")
        extract_nested_archives(output_dir, protected_files)
        
        print(f"\n🔒 Main archive preserved: {archive_path.name}")
    else:
        print(f"❌ Failed to extract main archive")
        return
    
    print("=" * 70)
    print(f"✅ Extraction complete!")
    print(f"📂 Results saved to: {output_dir}")
    print("=" * 70)


def extract_batch_archives(directory, output_dir=None):
    """
    Extract multiple archive files from a directory (non-recursive directory scan).
    Each archive is extracted to its own folder.
    The main archives in the directory are NOT deleted, only nested archives are removed.
    
    Args:
        directory: Directory containing archive files
        output_dir: Output directory (if None, creates folders next to archives)
    """
    directory = Path(directory)
    
    if not directory.is_absolute():
        directory = directory.expanduser()
    else:
        directory = directory.resolve()
    
    if not directory.exists():
        raise FileNotFoundError(f"Directory does not exist: {directory}")
    
    if not directory.is_dir():
        raise ValueError(f"Path is not a directory: {directory}")
    
    # Determine output directory
    if output_dir is None:
        output_base = directory
    else:
        output_base = Path(output_dir).expanduser().resolve()
        output_base.mkdir(parents=True, exist_ok=True)
    
    print("=" * 70)
    print("🚀 BATCH ARCHIVE EXTRACTOR")
    print("=" * 70)
    print(f"📂 Input Directory:  {directory}")
    print(f"📂 Output Directory: {output_base}")
    print(f"🔒 Main archives will be preserved")
    print("=" * 70)
    
    # Find all archives in the directory (only top level, not recursive)
    archives = [f for f in directory.iterdir() if f.is_file() and is_archive(f)]
    
    if not archives:
        print("⚠️  No archives found in the directory")
        return
    
    print(f"\n📊 Found {len(archives)} archive(s) to process\n")
    
    # Protect all main archives from deletion
    protected_files = {archive.resolve() for archive in archives}
    
    # Process each archive
    for idx, archive_path in enumerate(archives, 1):
        print(f"\n{'─' * 70}")
        print(f"Processing [{idx}/{len(archives)}]: {archive_path.name}")
        print(f"{'─' * 70}")
        
        # Create output folder for this archive
        folder_name = get_archive_stem(archive_path)
        extract_folder = output_base / folder_name
        
        # Extract the archive
        print(f"📦 Extracting: {archive_path.name} → {folder_name}/")
        if extract_archive(archive_path, extract_folder):
            print(f"✅ Archive extracted successfully")
            
            # Extract all nested archives within this extraction
            print(f"🔍 Searching for nested archives...")
            extract_nested_archives(extract_folder, protected_files)
            
            print(f"🔒 Main archive preserved: {archive_path.name}")
        else:
            print(f"❌ Failed to extract: {archive_path.name}")
    
    print("\n" + "=" * 70)
    print(f"✅ Batch extraction complete!")
    print(f"📂 Results saved to: {output_base}")
    print("=" * 70)


Extract from a zip file

In [16]:
# Extract single archive (deletes nested archives after extraction)
extract_single_archive("/Users/apple/Downloads/welldata/New Zealand/Tawaki-1.zip")

🚀 SINGLE ARCHIVE EXTRACTOR
📂 Input:  /Users/apple/Downloads/welldata/New Zealand/Tawaki-1.zip
📂 Output: /Users/apple/Downloads/welldata/New Zealand/Tawaki-1
🔒 Main archive will be preserved
📦 Extracting main archive: Tawaki-1.zip
✅ Main archive extracted successfully

🔍 Searching for nested archives...
📦 Extracting: nz-tawhaki-1-all/pr5842l3.zip → pr5842l3/
   🗑️  Deleted: pr5842l3.zip
📦 Extracting: nz-tawhaki-1-all/pr5842l2.zip → pr5842l2/
   🗑️  Deleted: pr5842l2.zip
📦 Extracting: nz-tawhaki-1-all/pr5842l1.zip → pr5842l1/
   🗑️  Deleted: pr5842l1.zip
📦 Extracting: nz-tawhaki-1-all/pr5842l4.zip → pr5842l4/
   🗑️  Deleted: pr5842l4.zip

🔒 Main archive preserved: Tawaki-1.zip
✅ Extraction complete!
📂 Results saved to: /Users/apple/Downloads/welldata/New Zealand/Tawaki-1


Extract from a directory containing archives

In [ ]:
# Extract all archives in a directory
extract_batch_archives("/path/to/directory")